# Проверка интеграции MVP / R1 — Run All

После обновления кода: **Restart Kernel → Run All**, чтобы не использовать старые импортированные модули.

Это **не обучение** и не подбор порога. Все этапы идут до завершения, без общего лимита времени. Новый каталог создаётся при каждом запуске; старые результаты не перезаписываются. По умолчанию — CPU и полная входная выборка. MVP остаётся активным независимо от результата.

Окружение: Python 3.11, `requirements-verification.txt`, Docker, Node/npm, установленные `web-ui/node_modules` и Chromium для Playwright. Выберите kernel новой `.venv`. При первом запуске подготовьте зависимости по `docs/RELEASE_INTEGRATION.md`. Исходные research/app нужны только для независимого эталонного сравнения; конкурсному runtime они не нужны.

БД проверки создаётся отдельно и удаляется после теста. Рабочая БД не используется. Сборка Docker может использовать сеть; контейнеры инференса запускаются с `--network none`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "backend/runtime.py").is_file():
    ROOT = ROOT.parent
assert (ROOT / "backend/runtime.py").is_file(), "Откройте notebook из новой release-integration копии"
sys.path.insert(0, str(ROOT))
from tools.release_verify import Verification

RESEARCH = ROOT.parent / "Car-classification-MSK"
LEGACY_APP = ROOT.parent / "Car-classification-MSK-review-fixes"
DATASET = ROOT / "dataset" if (ROOT / "dataset").is_dir() else RESEARCH / "dataset"
REFERENCE_PYTHON = RESEARCH / ".venv/bin/python"
PROVIDER = "CPUExecutionProvider"  # На GPU: CUDAExecutionProvider. Fallback запрещён.
SMOKE = False                     # False = вся выборка; общего ограничения времени нет.

run = Verification(DATASET, research=RESEARCH, legacy_app=LEGACY_APP,
                   reference_python=REFERENCE_PYTHON, provider=PROVIDER, smoke=SMOKE)
print("Результаты:", run.output)
print("Kernel:", sys.executable)

## 1. Preflight и компоненты

Проверяются все файлы каталога весов, frozen-профили, входные CSV/изображения, bbox, ошибки, контракт экспорта и отсутствие runtime-зависимости от обучения/БД/сети.

In [ ]:
run.stage("preflight", run.preflight)
run.stage("components", run.components)

## 2. Полные экспорты и независимый эталон

Выполняются новые экспорты MVP и ансамбля R1. Сравниваются float32-векторы, конкретные top-10, ID кандидатов и отказы. Допуск векторов фиксирован: 2e-4; изменения решений не допускаются. Скорость старых runs не используется.

In [ ]:
run.stage("exports_reference", run.exports_and_reference)

## 3. Независимость query и откат

Все query отдельно и с batch 8/16/32; перестановка и удаление соседних query. Галерея неизменна. Проверяется MVP → R1 → MVP и точное сохранение решений после загрузки кэша.

In [ ]:
run.stage("streaming_rollback", run.streaming_and_rollback)

## 4. Тестовая PostgreSQL, API, интерфейс и offline Docker

Отдельные пространства 512/1536; атомарность неуспешной сборки; старые таблицы не изменяются. API-тесты входят в components. UI проверяет в том числе raw-кандидата, отличающегося от ranking top-1.

In [ ]:
run.stage("postgres", run.postgres)
run.stage("ui", run.ui)
run.stage("docker_offline", run.docker)

## 5. Новые измерения скорости

Каждый профиль: 50 прогревов, 300 latency batch-1, throughput batch 1/8/16/32 не менее 10 секунд каждый, новый полный extract. Учитываются decode/crop/preprocess/transfers/forward/L2. Полное время экспорта измерено отдельно. RAM и NVML/VRAM — выборочные измерения, не гарантированный абсолютный пик. Mac не подтверждает RTX A5000 или native Linux amd64.

In [ ]:
run.stage("fresh_benchmarks", run.benchmarks)

## 6. Решение и отчёт

Даже если этап завершился ошибкой, остальные проверки выполняются и сохраняют результаты. Итог явно указывает проваленные проверки; повышение допуска и изменение threshold запрещены. R1 не становится активным автоматически.

In [ ]:
decision = run.finish()
print("Отчёт:", run.output / "REPORT.md")
print("Подробности:", run.output / "release_decision.json")
if not decision["local_acceptance_passed"]:
    raise RuntimeError("Не все проверки прошли. Отчёт сохранён; MVP остаётся активным.")